# GenAI Assignment 2: Clinical Trial Eligibility with Transformer Inference Mechanisms

**Course:** Generative AI
**Author:** Muhammad Hamza

---

This notebook runs the complete pipeline:
1. **Task 1:** Dataset preparation, model selection & QLoRA fine-tuning
2. **Task 2:** Comprehensive evaluation with classification + evidence metrics
3. **Task 3:** KV-Cached autoregressive generation
4. **Task 4:** Tiled attention with online softmax (FlashAttention-style)
5. **Task 5:** Speculative decoding
6. **Task 6:** Interactive Streamlit application

**Hardware:** Google Colab T4 GPU recommended

In [ ]:
# ============================================================
# Environment Setup
# ============================================================
!pip install -q torch transformers datasets peft bitsandbytes accelerate trl
!pip install -q numpy pandas scikit-learn matplotlib seaborn tqdm
!pip install -q streamlit plotly sentencepiece protobuf huggingface_hub pyyaml

import os, sys
# On Google Colab, clone / pull latest repository code
if os.path.exists('/content'):
    !rm -rf temp_repo
    !git clone -q https://github.com/hamzaa112233/GenAI-Assignment2.git temp_repo
    !cp -rf temp_repo/* .
    !rm -rf temp_repo

sys.path.insert(0, '.')
os.makedirs('reports/figures', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


---
## Task 1a: Dataset Preparation & Patient-Grouped Splitting

Download `ncbi/TrialGPT-Criterion-Annotations`, create strict patient-grouped splits (seed=42, 37/8/8), remove GPT-4 leakage columns, and save split manifest.

In [ ]:
from src.data.dataset import prepare_datasets

ds_dict, patient_splits = prepare_datasets(output_dir='./outputs')

print(f"\nTrain: {len(ds_dict['train'])} samples")
print(f"Validation: {len(ds_dict['validation'])} samples")
print(f"Test: {len(ds_dict['test'])} samples")

# Show a sample
sample = ds_dict['train'][0]
print(f"\n--- Sample ---")
print(f"Patient ID: {sample['patient_id']}")
print(f"Input (first 300 chars): {sample['input'][:300]}...")
print(f"Output: {sample['output']}")

## Task 1b: Model Architecture Selection

We compared:
- **Qwen2.5-1.5B-Instruct** (selected): Best instruction-following, robust JSON output, compatible draft model available
- **Llama-3.2-1B-Instruct**: Competitive but weaker structured output
- **SmolLM2-1.7B-Instruct**: Similar size but weaker clinical reasoning

### Load Model with QLoRA

In [ ]:
from src.models.model import load_model_for_training

model, tokenizer = load_model_for_training(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    lora_r=16, lora_alpha=32, lora_dropout=0.05
)

## Task 1c: Fine-Tuning

In [ ]:
import importlib
import src.models.train as train_module
importlib.reload(train_module)
from src.models.train import main as train_main

# Run the full training pipeline (reuses the model & tokenizer loaded above)
train_results = train_main(config_path='configs/training_config.yaml', model=model, tokenizer=tokenizer)


In [ ]:
# Display training curves
from IPython.display import Image, display
display(Image(filename='reports/figures/training_curves.png'))

---
## Task 2: Criterion-Level Eligibility Evaluation

Evaluate on the held-out test set with:
- Multi-class classification metrics (Macro-F1, Accuracy, per-class P/R/F1)
- Evidence sentence retrieval metrics (Micro/Macro P/R/F1)
- Confusion matrices (overall, inclusion, exclusion)
- Error analysis (8 distinct categories)

In [ ]:
from src.evaluation.evaluate import run_full_evaluation

eval_results = run_full_evaluation(
    adapter_path='outputs/best_adapter',
    base_model_name='Qwen/Qwen2.5-1.5B-Instruct',
    output_dir='reports'
)

In [ ]:
# Display confusion matrices
from IPython.display import Image, display
import os

for fig_name in ['confusion_matrix_overall.png', 'confusion_matrix_inclusion.png', 'confusion_matrix_exclusion.png']:
    path = f'reports/figures/{fig_name}'
    if os.path.exists(path):
        print(f'\n{fig_name}:')
        display(Image(filename=path))

---
## Task 3: KV-Cached Autoregressive Generation

Implement and compare:
- **Uncached**: Full recomputation at every decode step
- **KV-Cached**: Manual past_key_values management, only new token fed per step

Verify token-level equivalence and benchmark performance.

In [ ]:
from src.inference.kv_cache import verify_equivalence, benchmark

# Verify token-level equivalence across 3 different sequence lengths
print('='*60)
print('STEP 1: Token-Level Equivalence Verification')
print('='*60)
equiv = verify_equivalence(model_name='Qwen/Qwen2.5-1.5B-Instruct')

In [ ]:
# Benchmark KV-cached vs uncached generation
print('\n' + '='*60)
print('STEP 2: Performance Benchmark')
print('='*60)
kv_results = benchmark(model_name='Qwen/Qwen2.5-1.5B-Instruct')

from IPython.display import Image, display
display(Image(filename='reports/figures/kv_cache_benchmark.png'))

---
## Task 4: Tiled Attention with Online Softmax

Implement FlashAttention-style blocked computation:
- Standard SDPA reference with causal masking and stable softmax
- Tiled online-softmax attention with running max (m), normalizer (l), and output rescaling
- Validate numerical agreement within 1e-4 tolerance

In [ ]:
from src.inference.tiled_attention import validate_numerical_agreement, benchmark_attention

# Validate numerical agreement across multiple shapes
print('='*60)
print('STEP 1: Numerical Agreement Validation')
print('='*60)
agreement = validate_numerical_agreement()

In [ ]:
# Benchmark runtime and memory
print('\n' + '='*60)
print('STEP 2: Runtime & Memory Benchmark')
print('='*60)
attn_results = benchmark_attention()

from IPython.display import Image, display
display(Image(filename='reports/figures/tiled_attention_benchmark.png'))

---
## Task 5: Greedy Speculative Decoding

- Draft model: Qwen2.5-0.5B-Instruct
- Target model: Qwen2.5-1.5B-Instruct
- K=5 candidate tokens per iteration
- Verify token equivalence to standard greedy target-only decoding

In [ ]:
from src.inference.speculative import verify_equivalence as spec_verify, benchmark as spec_benchmark

# Verify token-level equivalence
print('='*60)
print('STEP 1: Token Equivalence Verification')
print('='*60)
spec_equiv = spec_verify(
    draft_model_name='Qwen/Qwen2.5-0.5B-Instruct',
    target_model_name='Qwen/Qwen2.5-1.5B-Instruct'
)

In [ ]:
# Benchmark speculative vs standard generation
print('\n' + '='*60)
print('STEP 2: Speculative Decoding Benchmark')
print('='*60)
spec_results = spec_benchmark(
    draft_model_name='Qwen/Qwen2.5-0.5B-Instruct',
    target_model_name='Qwen/Qwen2.5-1.5B-Instruct'
)

from IPython.display import Image, display
display(Image(filename='reports/figures/speculative_benchmark.png'))

---
## Task 6: Streamlit Application

Launch the interactive browser application. On Colab, use localtunnel for external access.

In [ ]:
# To launch the Streamlit app on Colab:
# Method 1: Direct launch (works in local Jupyter)
# !streamlit run src/app/streamlit_app.py

# Method 2: Colab with localtunnel
# !npm install -g localtunnel
# import subprocess
# proc = subprocess.Popen(['streamlit', 'run', 'src/app/streamlit_app.py', '--server.port', '8501'])
# !npx localtunnel --port 8501

print('To launch the Streamlit app:')
print('  streamlit run src/app/streamlit_app.py')
print('\nThe app includes:')
print('  - Eligibility assessment with evidence highlighting')
print('  - Inference benchmark dashboard')
print('  - Report export functionality')
print('  - Clinical disclaimer banner')

---
## Summary: Save All Results

In [ ]:
import json

all_results = {
    'classification_metrics': eval_results.get('classification', {}),
    'evidence_metrics': eval_results.get('evidence', {}),
    'kv_cache_benchmark': kv_results,
    'attention_benchmark': attn_results,
    'speculative_benchmark': spec_results
}

with open('reports/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print('All results saved to reports/all_results.json')
print('\n=== ASSIGNMENT COMPLETE ===')